# Tome 0, Chapter 3: Polynomials (Python companion)

Exercises first, solutions at the end. Computer-science threads: Horner's rule, interpolation in floating point
(Runge vs Chebyshev), Newton's method, and **Shamir secret sharing**, a cryptographic protocol that is nothing but Lagrange interpolation.

In [ ]:
import random
from fractions import Fraction
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

## Exercises

**P3.1 (★) Horner's rule.** Evaluate $P(x) = \sum_k a_k x^k$ with only $n$ multiplications by writing $P(x) = a_0 + x(a_1 + x(a_2 + \cdots))$. Implement it, and use it to recover the remainder of Exercise 3.1 with `sp.div`.

In [ ]:
# Your code here


**P3.2 (★) Viète by machine.** Check Exercise 3.2 numerically with `np.roots`, then symbolically with SymPy's `sp.polys.polyfuncs.symmetrize`, which rewrites a symmetric polynomial in terms of the $\sigma_k$.

In [ ]:
# Your code here


**P3.3 (★★) Runge's phenomenon.** Interpolate $f(x) = \frac{1}{1+25x^2}$ on $[-1,1]$ at $n+1$ equally spaced points and at the $n+1$ Chebyshev nodes $\cos\frac{(2k+1)\pi}{2n+2}$, for $n = 5, 10, 20$. Plot both and print the maximum errors. Which grows, which shrinks? Relate to Exercise 3.6.

In [ ]:
# Your code here


**P3.4 (★★) Newton's fractal.** Newton's method $z \mapsto z - P(z)/P'(z)$ for $P = z^3 - 1$ converges to one of the three cube roots of unity. Colour each starting point of $[-1.5, 1.5]^2$ by the root it reaches. What do the boundaries look like?

In [ ]:
# Your code here


**P3.5 (★★) CS application: Shamir secret sharing.** Work modulo the prime $p = 2^{127}-1$. To share a secret $s$ among $n$ people so that any $k$ of them can recover it but $k-1$ learn nothing:
1. choose a random polynomial $P$ of degree $k-1$ over $\mathbb Z/p\mathbb Z$ with $P(0) = s$;
2. give person $i$ the share $(i, P(i))$;
3. any $k$ shares determine $P$ by Lagrange (Theorem 3.5), hence $s = P(0)$.

Implement `share(s, k, n)` and `recover(shares)`, and test with $k = 3$, $n = 5$. Why do $k - 1$ shares reveal nothing about $s$?

In [ ]:
# Your code here


**P3.6 (★★★) Basel, checked.** Verify the identity $\sum_{k=1}^m \cot^2\frac{k\pi}{2m+1} = \frac{m(2m-1)}{3}$ numerically for $m \le 50$, and plot the two bounds of Exercise 3.7(c) closing in on $\pi^2/6$.

In [ ]:
# Your code here


---
## Solutions

### P3.1

In [ ]:
def horner(coeffs, x):
    """coeffs = [a_0, a_1, ..., a_n]."""
    acc = 0
    for a in reversed(coeffs):
        acc = acc * x + a
    return acc

assert horner([1, 2, 3], 10) == 321
X = sp.symbols("X")
for n in range(1, 8):
    q, r = sp.div(X**n, (X - 1) * (X - 2), X)
    assert sp.expand(r - ((2**n - 1) * X + 2 - 2**n)) == 0
print("Horner works; remainder formula of Exercise 3.1 confirmed for n <= 7")

### P3.2

In [ ]:
r = np.roots([1, 0, -1, 1])
print("sum x^2 =", np.round(np.sum(r**2), 12), "  sum x^3 =", np.round(np.sum(r**3), 12))
x1, x2, x3 = sp.symbols("x1 x2 x3")
print(sp.polys.polyfuncs.symmetrize(x1**3 + x2**3 + x3**3, formal=True))

SymPy returns $s_1^3 - 3s_1s_2 + 3s_3$; with $s_1 = 0$, $s_3 = -1$ this is $-3$.

### P3.3

In [ ]:
f = lambda x: 1 / (1 + 25 * x**2)
xs = np.linspace(-1, 1, 2001)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, n in zip(axes, [5, 10, 20]):
    eq = np.linspace(-1, 1, n + 1)
    ch = np.cos((2 * np.arange(n + 1) + 1) * np.pi / (2 * n + 2))
    for nodes, label in [(eq, "equispaced"), (ch, "Chebyshev")]:
        poly = np.polynomial.polynomial.Polynomial.fit(nodes, f(nodes), n)
        err = np.max(np.abs(poly(xs) - f(xs)))
        ax.plot(xs, poly(xs), label=f"{label} (max err {err:.2g})")
    ax.plot(xs, f(xs), "k--", lw=1, label="f")
    ax.set_ylim(-0.5, 1.5); ax.set_title(f"n = {n}"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

The interpolation error is $\frac{f^{(n+1)}(\xi)}{(n+1)!}\prod_i (x - x_i)$. The product $\prod_i (x-x_i)$ is monic of degree $n+1$; Chebyshev nodes make it $2^{-n}T_{n+1}$, the smallest possible on $[-1,1]$ by Exercise 3.6. Equispaced nodes make it huge near the ends: the error explodes.

### P3.4

In [ ]:
N = 600
y, x = np.mgrid[-1.5:1.5:N * 1j, -1.5:1.5:N * 1j]
z = x + 1j * y
with np.errstate(all="ignore"):
    for _ in range(40):
        z = z - (z**3 - 1) / (3 * z**2)
roots = np.exp(2j * np.pi * np.arange(3) / 3)
basin = np.argmin(np.abs(z[..., None] - roots), axis=-1)
plt.figure(figsize=(6, 6))
plt.imshow(basin, extent=(-1.5, 1.5, -1.5, 1.5), cmap="viridis", origin="lower")
plt.title("Basins of attraction of Newton's method for $z^3 - 1$"); plt.axis("off"); plt.show()

The boundary between basins is a fractal (a Julia set): at every boundary point, all three colours meet.

### P3.5

In [ ]:
P = 2**127 - 1  # a Mersenne prime

def share(secret, k, n, rng=random.SystemRandom()):
    coeffs = [secret] + [rng.randrange(P) for _ in range(k - 1)]
    poly = lambda x: sum(c * pow(x, e, P) for e, c in enumerate(coeffs)) % P
    return [(i, poly(i)) for i in range(1, n + 1)]

def recover(shares):
    """Lagrange interpolation at 0, modulo P."""
    s = 0
    for i, (xi, yi) in enumerate(shares):
        num, den = 1, 1
        for j, (xj, _) in enumerate(shares):
            if j != i:
                num = num * (-xj) % P
                den = den * (xi - xj) % P
        s = (s + yi * num * pow(den, -1, P)) % P
    return s

secret = 271828182845904523536
shares = share(secret, k=3, n=5)
from itertools import combinations
assert all(recover(list(c)) == secret for c in combinations(shares, 3))
print("every 3 of the 5 shares recover the secret")
print("2 shares give:", recover(shares[:2]), "(garbage)")

With $k-1$ shares, for **every** candidate secret $s'$ there is exactly one polynomial of degree $k-1$ through those shares and $(0, s')$ (Lagrange again). All secrets are equally likely: the shares carry no information. This is perfect secrecy, not just computational hardness.

### P3.6

In [ ]:
for m in range(1, 51):
    s = sum(1 / np.tan(k * np.pi / (2 * m + 1)) ** 2 for k in range(1, m + 1))
    assert np.isclose(s, m * (2 * m - 1) / 3)
ms = np.arange(1, 200)
low = np.pi**2 / (2 * ms + 1) ** 2 * ms * (2 * ms - 1) / 3
high = low + np.pi**2 / (2 * ms + 1) ** 2 * ms
plt.plot(ms, low, label="lower bound"); plt.plot(ms, high, label="upper bound")
plt.axhline(np.pi**2 / 6, color="k", ls="--", label=r"$\pi^2/6$")
plt.xscale("log"); plt.legend(); plt.title("Squeezing $\\zeta(2)$"); plt.show()